In [2]:
import numpy as np
import geopandas as gpd
import pandas as pd
from shapely.geometry import Polygon, MultiPolygon
import folium
from folium.features import GeoJsonTooltip
rng = np.random.default_rng(42)  # reproducibility
from pathlib import Path

In [11]:
# # Drop patches with errors
# fire_patches = gpd.read_parquet("../data/CCC-ImpactMesh-Fire/patch_metadata.parquet").reset_index(drop=True)
#
# import re
# error_patches = ["EMSR408_7_56JLL_x406835*", "EMSR765_13_20KRG_x883425*", "EMSR837_4_29TPH_x701805*", "EMSR760_4_29TNG_*_y4603685", "EMSR647_1_18HXD_*_y5876605", "EMSR765_6_20LLJ_x396015_*", "EMSR765_9_20LKH_x307005_y8242915"]
#
#
#
# def wildcard_to_regex(pattern):
#     pattern = re.escape(pattern)
#     pattern = pattern.replace(r'\*', '.*')
#     return f'^{pattern}$'
#
# regex_patterns = [wildcard_to_regex(p) for p in error_patches]
# combined_pattern = '|'.join(regex_patterns)
#
# fire_patches = fire_patches[~fire_patches['patch_id'].str.match(combined_pattern, na=False)]
# fire_patches.to_parquet("../data/CCC-ImpactMesh-Fire/patch_metadata.parquet")

In [2]:
fire = gpd.read_parquet("../data/CCC-ImpactMesh-Fire/tile_metadata.parquet").reset_index(drop=True)
fire_patches = gpd.read_parquet("../data/CCC-ImpactMesh-Fire/patch_metadata.parquet").reset_index(drop=True)
len(fire_patches)

33968

In [3]:
# Join chunked processing
fire = fire.sort_values(['S2_num_images', 'S1_num_images'], ascending=False).sort_values(['event_id'])
counts = fire.groupby('event_id')[['pixel_ignore', 'pixel_background', 'pixel_wildfire', 'num_patches']].sum()
fire = fire.drop_duplicates('event_id')
fire[['pixel_ignore', 'pixel_background', 'pixel_wildfire', 'num_patches']] = counts.loc[fire.event_id].values

In [4]:
# Drop missing event windows
fire_patches = fire_patches[~fire_patches.S1_event_window_date.isnull()]
fire_patches = fire_patches[~fire_patches.S2_event_window_date.isnull()]
len(fire_patches)

31010

In [5]:
# Reduce to events AOIs
fire_events = fire.drop_duplicates(["code", "aoi_number"]).drop(columns=["event_id"])
# Add num patches
fire_events.set_index(["code", "aoi_number"], drop=True, inplace=True)
fire_events["num_patches"] = fire_patches.value_counts(["code", "aoi_number"])
fire_events["num_pos_patches"] = fire_patches[fire_patches["positive_sample"]].value_counts(["code", "aoi_number"])
fire_events["num_pos_patches"] = fire_events["num_pos_patches"].fillna(0).astype(int)
fire_events.reset_index(inplace=True)

In [6]:
fire_events

,code,aoi_number,title,location,aoi_name,event_type,subcategory,event_date,annotation_date,aoi,...,S2_pre_month_cloud_coverage,S2_pre_event_cloud_coverage,S2_event_window_cloud_coverage,S2_post_event_cloud_coverage,pixel_ignore,pixel_background,pixel_wildfire,split,num_patches,num_pos_patches
0,EMSR168,1,Forest Fire in Cyprus,Cyprus,Mokounta,Wildfire,,2016-06-18 00:00:00,2016-06-21,"POLYGON ((32.54458 35.04007, 32.49596 35.04007...",...,0.000208,0.000104,0.000166,0.019324,5982,227824,151410,None,2.0,2
1,EMSR168,2,Forest Fire in Cyprus,Cyprus,Argaka,Wildfire,,2016-06-18 00:00:00,2016-06-21,"POLYGON ((32.5662 35.01132, 32.46335 35.01132,...",...,0.000908,0.001871,0.093665,0.057853,19010,1749676,151410,None,18.0,10
2,EMSR168,3,Forest Fire in Cyprus,Cyprus,Evrychou,Wildfire,,2016-06-18 00:00:00,2016-06-23,"POLYGON ((33.02067 35.08287, 33.02067 34.92447...",...,0.000588,0.006447,0.001274,0.006661,3890,2670103,194936,None,36.0,9
3,EMSR168,4,Forest Fire in Cyprus,Cyprus,Sina Oros,Wildfire,,2016-06-18 00:00:00,2016-06-23,"POLYGON ((32.95225 35.02798, 32.95225 34.97906...",...,0.000268,0.000238,0.000074,0.000863,256,101009,167520,None,2.0,2
4,EMSR169,10,Fires in Sicily,Italy,Pantelleria,Wildfire,,2016-06-16 00:00:00,2016-06-25,"POLYGON ((12.06626 36.72216, 11.91567 36.72216...",...,0.000000,0.002937,0.002841,0.000000,123350,1886452,66961,None,25.0,7
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
489,EMSR842,2,Wildfire in Spain and Portugal,Spain,Pendilla de Arbas,Wildfire,Forest fire,2025-09-08 19:00:00,2025-09-24,"POLYGON ((-5.65065 43.04981, -5.63996 42.99048...",...,0.000000,0.000000,0.000000,0.000000,98900,574261,31647,None,6.0,3
490,EMSR859,1,"Wildfires in the Biobio and Nuble Regions, Chile",Chile,San Rosendo,Wildfire,Forest fire,2026-01-16 19:00:00,2026-01-21,"POLYGON ((-72.62 -37.217, -72.574 -37.276, -72...",...,0.000000,0.000001,0.000158,0.000000,1123775,2430775,278244,None,33.0,16
491,EMSR859,2,"Wildfires in the Biobio and Nuble Regions, Chile",Chile,Chiguaihue,Wildfire,Forest fire,2026-01-16 19:00:00,2026-01-25,"POLYGON ((-72.73336 -37.87816, -72.68225 -37.8...",...,0.000000,0.000000,0.000019,0.000000,3552419,5004237,331114,None,119.0,50
492,EMSR859,3,"Wildfires in the Biobio and Nuble Regions, Chile",Chile,Santa Barbara,Wildfire,Forest fire,NaT,2026-01-24,"POLYGON ((-72.5757 -36.85286, -72.5881 -36.857...",...,0.000000,0.000000,0.031288,0.000024,10899957,11384933,880566,None,243.0,45


# Group overlapping events

In [17]:
def make_global_groups(gdf, alpha=0.1):
    assert alpha < 0.5
    gdf = gdf.copy().reset_index()
    buf = gdf.copy()
    buf = buf.to_crs(3857)

    area = buf.geometry.area  # m^2
    r_eff = np.sqrt(area / np.pi)  # effective radius in meters

    # Hyperparameters: shrink factor, and optional clamps to be safe
    dist = alpha * r_eff

    buf["geometry"] = buf.geometry.buffer(-dist)
    buf = buf.set_geometry("geometry")

    merged = buf.union_all()
    comps = [merged] if isinstance(merged, Polygon) else list(merged.geoms)
    groups = gpd.GeoDataFrame({"group_id": np.arange(len(comps))},
                              geometry=comps, crs=3857)

    joined = gpd.overlay(buf, groups, how="intersection")
    joined["overlap_area"] = joined.geometry.area
    joined = joined.sort_values(["index", "overlap_area"]).drop_duplicates("index", keep="last")
    gdf["group_id"] = joined["group_id"].values
    groups = groups.to_crs(4326)

    for g, group in gdf.groupby('group_id'):
        groups.at[g, "geometry"] = group.union_all()
    print(len(groups), "groups")
    return gdf, groups

In [18]:
gdf, groups = make_global_groups(fire_events, alpha=0.1)
# Filter groups by ones that are actually selected
gdf["num_aoi"] = 1
groups[["num_patches", "num_pos_patches", "num_aoi"]] = gdf[["group_id", "num_patches", "num_pos_patches", "num_aoi"]].groupby("group_id").sum()
groups["num_patches"] = groups["num_patches"].clip(0, 500)
groups["num_pos_patches"] = groups["num_pos_patches"].clip(0, 200)
groups = groups[~groups.num_aoi.isna()]
groups.reset_index(drop=True, inplace=True)
# Reset group ids
mapping = {g: i for i, g in groups.group_id.items()}
groups["group_id"] = groups.apply(lambda x: mapping[x["group_id"]], axis=1)
gdf["group_id"] = gdf.apply(lambda x: mapping[x["group_id"]], axis=1)

306 groups


In [19]:
gdf

,index,code,aoi_number,title,location,aoi_name,event_type,subcategory,event_date,annotation_date,...,S2_event_window_cloud_coverage,S2_post_event_cloud_coverage,pixel_ignore,pixel_background,pixel_wildfire,split,num_patches,num_pos_patches,group_id,num_aoi
0,0,EMSR168,1,Forest Fire in Cyprus,Cyprus,Mokounta,Wildfire,,2016-06-18 00:00:00,2016-06-21,...,0.000166,0.019324,5982,227824,151410,None,2.0,2,278,1
1,1,EMSR168,2,Forest Fire in Cyprus,Cyprus,Argaka,Wildfire,,2016-06-18 00:00:00,2016-06-21,...,0.093665,0.057853,19010,1749676,151410,None,18.0,10,278,1
2,2,EMSR168,3,Forest Fire in Cyprus,Cyprus,Evrychou,Wildfire,,2016-06-18 00:00:00,2016-06-23,...,0.001274,0.006661,3890,2670103,194936,None,36.0,9,276,1
3,3,EMSR168,4,Forest Fire in Cyprus,Cyprus,Sina Oros,Wildfire,,2016-06-18 00:00:00,2016-06-23,...,0.000074,0.000863,256,101009,167520,None,2.0,2,276,1
4,4,EMSR169,10,Fires in Sicily,Italy,Pantelleria,Wildfire,,2016-06-16 00:00:00,2016-06-25,...,0.002841,0.000000,123350,1886452,66961,None,25.0,7,139,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
489,489,EMSR842,2,Wildfire in Spain and Portugal,Spain,Pendilla de Arbas,Wildfire,Forest fire,2025-09-08 19:00:00,2025-09-24,...,0.000000,0.000000,98900,574261,31647,None,6.0,3,83,1
490,490,EMSR859,1,"Wildfires in the Biobio and Nuble Regions, Chile",Chile,San Rosendo,Wildfire,Forest fire,2026-01-16 19:00:00,2026-01-21,...,0.000158,0.000000,1123775,2430775,278244,None,33.0,16,3,1
491,491,EMSR859,2,"Wildfires in the Biobio and Nuble Regions, Chile",Chile,Chiguaihue,Wildfire,Forest fire,2026-01-16 19:00:00,2026-01-25,...,0.000019,0.000000,3552419,5004237,331114,None,119.0,50,0,1
492,492,EMSR859,3,"Wildfires in the Biobio and Nuble Regions, Chile",Chile,Santa Barbara,Wildfire,Forest fire,NaT,2026-01-24,...,0.031288,0.000024,10899957,11384933,880566,None,243.0,45,4,1


# Create cells for sampling

In [20]:
# Params
cell_km = 150          # start coarse; increase to make bigger cells
min_groups_per_cell = 5

# 1) Project & coarse grid index
split = groups.to_crs(3857).copy()
L = cell_km * 1000.0
cent = split.geometry.centroid
gx, gy = (np.floor(cent.x.values / L).astype(int),
          np.floor(cent.y.values / L).astype(int))
split["cell0"] = list(zip(gx, gy))
# 2) Merge undersized cells into nearest sufficiently large cell
sizes = split["cell0"].value_counts()
big_cells = list(set(sizes.index[sizes >= min_groups_per_cell]))
small_cells = [c for c, n in sizes.items() if n < min_groups_per_cell]
# Precompute cell centroids for distance-based merge
cell_centroids = split.groupby("cell0").geometry.apply(lambda g: g.union_all().centroid)
cell_xy = cell_centroids.apply(lambda p: (p.x, p.y)).to_dict()

cell_map = {c:c for c in sizes.index}  # mapping cell0 -> eff_cell

for c in small_cells:
    cx, cy = cell_xy[c]
    # nearest big cell by euclidean distance in meters
    if len(big_cells):
        bc_dist = list(map(lambda b: ((cell_xy[b][0]-cx)**2 + (cell_xy[b][1]-cy)**2) ** 0.5, big_cells))
        bc = big_cells[np.argmin(bc_dist)]
        if min(bc_dist) < L * 3:  # ~500km distance
            cell_map[c] = bc
        else:
            cell_map[c] = "__misc__"
    else:
        cell_map[c] = "__misc__"

split["eff_cell"] = split["cell0"].map(cell_map)

cell_id = {c: i for i, c in enumerate(split["eff_cell"].unique())}
split["cell_id"] = split["eff_cell"].map(cell_id)

In [21]:
import numpy as np
import pandas as pd
import geopandas as gpd
from shapely.ops import unary_union

def group_polygons_iterative(
    groups: gpd.GeoDataFrame,
    cell_km: float = 200,
    min_groups_per_cell: int = 3,
    max_merge_km: float = 1000.0,       # hard cap on centroid-to-centroid merge distance
    crs_m: int = 3857,                  # projected CRS in meters
    centroid_mode: str = "mean",        # "mean" (fast) or "union" (geometric centroid)
    max_passes: int = 100,              # safety stop
    assign_misc: bool = True,           # map unmergeable small clusters to "__misc__"
    enforce_diameter: bool = False      # ensure the final cluster diameter <= max_merge_km
) -> gpd.GeoDataFrame:
    """
    Iteratively merges undersized grid cells into their nearest neighbor (which may be small or big),
    respecting a hard maximum merge distance between cluster centroids. Recomputes cluster sizes and
    centroids after each merge. Optionally enforces a maximum within-cluster diameter (approximate).

    Returns a copy with columns: cell0, eff_cell, cell_id.
    """
    g = groups.to_crs(crs_m).copy().reset_index(drop=True)
    L = float(cell_km) * 1000.0
    MAX_D = float(max_merge_km) * 1000.0

    # --- 1) Initial grid cells (like your code) ---
    cent = g.geometry.centroid
    gx = np.floor(cent.x.values / L).astype(np.int64)
    gy = np.floor(cent.y.values / L).astype(np.int64)
    g["cell0"] = list(map(str, np.c_[gx, gy]))

    # Effective cell label (mutable during merging)
    g["eff_cell"] = g["cell0"]

    # Cache original point coordinates (to build fast centroids / diameter estimates)
    px = cent.x.values.astype(float)
    py = cent.y.values.astype(float)

    def _cluster_members(df: gpd.GeoDataFrame):
        """Return dict eff_cell -> numpy array of member row indices."""
        return {k: v.index.values for k, v in df.groupby("eff_cell", observed=True)}

    def _centroids(df: gpd.GeoDataFrame, mode="mean"):
        """Return dict eff_cell -> (x, y) in meters."""
        if mode == "mean":
            # mean of member centroids (fast, robust)
            out = {}
            for k, idx in _cluster_members(df).items():
                try:
                    out[k] = (float(px[idx].mean()), float(py[idx].mean()))
                except:
                    print(df)
            return out
        elif mode == "union":
            # geometric centroid of unary_union (accurate, slower)
            centroids = (
                df.groupby("eff_cell", observed=True)
                  .geometry.apply(lambda s: unary_union(s.values).centroid)
            )
            return {k: (p.x, p.y) for k, p in centroids.items()}
        else:
            raise ValueError("centroid_mode must be 'mean' or 'union'")

    def _approx_diameter(idx_union: np.ndarray, cx: float, cy: float) -> float:
        """
        Approximate cluster diameter as 2 * max radial distance from the cluster centroid.
        This is cheaper than O(n^2) pairwise distances, and errs on the safer side for compact shapes.
        """
        dx = px[idx_union] - cx
        dy = py[idx_union] - cy
        r = np.sqrt(dx * dx + dy * dy)
        return 2.0 * float(r.max()) if len(r) else 0.0

    passes = 0
    changed = True
    while changed and passes < max_passes:
        passes += 1
        changed = False

        # Current cluster sizes, labels, and centroids
        sizes = g["eff_cell"].value_counts()
        labels = list(sizes.index)
        small = [c for c, n in sizes.items() if n < 5]

        if not small:
            break

        # Centroids for all current clusters
        cxy = _centroids(g, centroid_mode)

        # Greedy: process the smallest clusters first; after each merge, restart the pass
        small_sorted = sorted(small, key=lambda c: sizes[c])
        did_merge_this_round = False

        for c in small_sorted:
            # Recompute guard: label might have been merged in an earlier iteration
            if c not in g["eff_cell"].unique():
                continue

            cx, cy = cxy[c]
            others = [o for o in labels if o != c]
            if not others:
                continue

            # Find nearest neighbor (can be small or big)
            bx = np.array([cxy[o][0] for o in others], dtype=float)
            by = np.array([cxy[o][1] for o in others], dtype=float)
            j = int(np.argmin((bx - cx) ** 2 + (by - cy) ** 2))
            target = others[j]
            tx, ty = cxy[target]
            d = float(np.hypot(tx - cx, ty - cy))

            if d <= MAX_D:
                # Optional: ensure merged cluster won't exceed max "cell diameter"
                if enforce_diameter:
                    members = _cluster_members(g)
                    idx_union = np.concatenate([members[c], members[target]])
                    # New centroid after merge (use same centroid mode; here we use mean for speed)
                    nx = float(np.r_[px[members[c]], px[members[target]]].mean())
                    ny = float(np.r_[py[members[c]], py[members[target]]].mean())
                    diam = _approx_diameter(idx_union, nx, ny)
                    if diam > MAX_D:
                        # Too spread out; try the next nearest neighbor instead.
                        # Remove this neighbor and attempt a different candidate.
                        # We do this by masking and searching the second-best, third-best, etc.
                        order = np.argsort((bx - cx) ** 2 + (by - cy) ** 2)
                        merged_ok = False
                        for jj in order:
                            cand = others[int(jj)]
                            tx2, ty2 = cxy[cand]
                            d2 = float(np.hypot(tx2 - cx, ty2 - cy))
                            if d2 > MAX_D:
                                continue
                            idx_union = np.concatenate([members[c], members[cand]])
                            nx2 = float(np.r_[px[members[c]], px[members[cand]]].mean())
                            ny2 = float(np.r_[py[members[c]], py[members[cand]]].mean())
                            diam2 = _approx_diameter(idx_union, nx2, ny2)
                            if diam2 <= MAX_D:
                                target = cand
                                d = d2
                                merged_ok = True
                                break
                        if not merged_ok:
                            continue  # give up on c for this pass

                # Perform the merge: point c into target (keep target label stable)
                g.loc[g["eff_cell"] == c, "eff_cell"] = target
                changed = True
                did_merge_this_round = True
                break  # restart a fresh pass (sizes/centroids have changed)

        if not did_merge_this_round:
            # No merges possible under the distance/diameter constraints
            break

    # Finalization: optionally map any still-small clusters to "__misc__"
    sizes = g["eff_cell"].value_counts()
    still_small = [c for c, n in sizes.items() if n < min_groups_per_cell]
    if assign_misc and len(still_small):
        g.loc[g["eff_cell"].isin(still_small), "eff_cell"] = "__misc__"

    # Stable numeric IDs
    eid = {c: i for i, c in enumerate(pd.unique(g["eff_cell"]))}
    g["cell_id"] = g["eff_cell"].map(eid)
    return g


split = group_polygons_iterative(groups)

In [22]:
split.value_counts("eff_cell")

eff_cell
[-4 25]      14
[ 4 24]      13
[-2 23]      13
[12 22]      13
[-4 22]      12
[14 23]      10
[ 84 -18]     9
[ 7 44]       9
[10 26]       8
[-4 26]       8
[-1 26]       8
[-50   9]     8
[-4 24]       8
[ 7 26]       8
[19 20]       8
[13 23]       7
[-41 -23]     7
[-35  -9]     7
[ 3 27]       7
[ 8 22]       7
[12 23]       6
[-9 16]       6
[13 25]       6
[ 0 25]       6
[-36 -10]     6
[ 8 24]       6
[ 5 26]       6
[ 7 22]       6
__misc__      6
[ 9 41]       6
[-1 25]       6
[11 24]       5
[11 25]       5
[ 7 34]       5
[12 25]       5
[-51   9]     5
[ 6 25]       5
[14 25]       5
[ 4 28]       5
[ 7 33]       5
[-3 40]       3
[12 35]       3
Name: count, dtype: int64

In [23]:
def map_from_gdf(gdf):
    m = folium.Map()
    folium.GeoJson(
        gdf,
        style_function=lambda x: {
            'color': 'black',
            'weight': 2,
            'fillColor': f'#{int(x["properties"]["cell_id"]) * 123456 % 0xFFFFFF:06x}',
            'fillOpacity': 0.6
        },
        tooltip=GeoJsonTooltip(fields=('group_id', 'num_patches', 'num_pos_patches', 'num_aoi', 'eff_cell', 'cell_id'))
    ).add_to(m)
    minx, miny, maxx, maxy = gdf.total_bounds
    m.fit_bounds([[miny/2, minx/2], [maxy/2, maxx/2]])
    return m

# Visalize effective cells
split["eff_cell"] = split["eff_cell"].astype(str).values
split = split.to_crs("EPSG:4326")
map_from_gdf(split[['group_id', 'geometry', 'num_patches', 'num_pos_patches', 'num_aoi', 'eff_cell', 'cell_id']])

# Sampling of groups within cells

In [24]:
# Params
target_ratio_patches = {"train": 0.66, "val": 0.16, "test": 0.18}
target_ratio_aois = {"train": 0.62, "val": 0.18, "test": 0.20}
seed=42
max_iters=5000

# 3) Per-cell sampling against targets (patches + AOIs), as discussed
splits = np.array(["train","val","test"])
p_patch = np.array([target_ratio_patches[s] for s in splits])
p_aoi   = np.array([target_ratio_aois[s]   for s in splits])
rng = np.random.default_rng(seed)

patches = split["num_patches"].astype(float).fillna(0.0).to_numpy()
aois    = split["num_aoi"].astype(float).fillna(0.0).to_numpy()
eff     = split["eff_cell"].astype(str).to_numpy()

idx_by_cell = {c: np.where(eff==c)[0] for c in np.unique(eff)}
assign_final = np.empty(len(split), dtype=int)

for c, idxs in idx_by_cell.items():
    best_loss, best = np.inf, None
    for _ in range(max_iters):
        a = rng.choice(3, size=len(idxs), p=p_patch)
        wp = np.bincount(a, weights=patches[idxs], minlength=3)
        wa = np.bincount(a, weights=aois[idxs],    minlength=3)
        rp = wp / max(wp.sum(), 1e-9)
        ra = wa / max(wa.sum(), 1e-9)
        loss = np.abs(rp - p_patch).sum() + np.abs(ra - p_aoi).sum()
        if loss < best_loss:
            best_loss, best = loss, a
    assign_final[idxs] = best

split["split"] = pd.Categorical(splits[assign_final], categories=list(splits), ordered=False)

In [25]:
# (Optional) quick diagnostics
global_summary = (split.assign(patches=patches, aois=aois)
                        .groupby("split")[["patches","aois"]].sum()
                        .assign(p_ratio=lambda d: d.patches/d.patches.sum(),
                                a_ratio=lambda d: d.aois/d.aois.sum()))
display(global_summary.head())

/var/folders/sr/4k1bv9r565d12nx315mt59_r0000gp/T/ipykernel_10768/3790876171.py:3: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  .groupby("split")[["patches","aois"]].sum()


,patches,aois,p_ratio,a_ratio
split,,,,
train,15929.0,317.0,0.666485,0.641700
val,3447.0,79.0,0.144226,0.159919
test,4524.0,98.0,0.189289,0.198381


# Map

In [26]:
split_colors = {
    "train": "#198038",  # dark green
    "val":   "#F09235",  # dark orange
    "test":  "#922752",  # dark red
}

def map_from_gdf(gdf):
    m = folium.Map()
    folium.GeoJson(
        gdf,
        style_function=lambda x: {
            'color': 'black',
            'weight': 2,
            # 'fillColor': f'#{int(x["properties"]["cell_id"]) * 123456 % 0xFFFFFF:06x}',
            'fillColor': split_colors[x["properties"]["split"]],
            'fillOpacity': 0.6
        },
        tooltip=GeoJsonTooltip(fields=("group_id", "eff_cell", "code", "aoi_name", "title", "num_patches"))
        # tooltip=GeoJsonTooltip(fields=('group_id', 'num_patches', 'num_pos_patches', 'num_aoi', 'cell_id'))
    ).add_to(m)
    minx, miny, maxx, maxy = gdf.total_bounds
    m.fit_bounds([[miny/2, minx/2], [maxy/2, maxx/2]])
    return m


# Events per split
gdf["split"] = split.loc[gdf.group_id]["split"].values
gdf["eff_cell"] = split.loc[gdf.group_id]["cell_id"].astype(str).values
map_from_gdf(gdf[["group_id", "eff_cell", "split", "code", "aoi_name", "title", "num_patches", "aoi"]])

In [27]:
splits = gdf.set_index(["code", "aoi_number"])

In [28]:
# Fire
fire["split"] = fire.apply(lambda row: splits["split"][(row["code"], row["aoi_number"])], axis=1)
fire_patches["split"] = fire_patches.apply(lambda row: splits["split"][(row["code"], row["aoi_number"])], axis=1)

In [29]:
# Drop all train patches that are less than 5 km from val/test area

# Get geometry of all test and val areas
val_test_poly = splits[splits["split"].isin(["val", "test"])].geometry.union_all()
val_test_poly = val_test_poly.buffer(0.05)
# Drop intersecting train samples
patches = fire_patches[~((fire_patches["split"] == "train") & fire_patches.geometry.intersects(val_test_poly))]
patches["negative_sample"] = patches["pixel_wildfire"] < 10
len(patches)

/Users/BLU/repos/impactmesh/venv/lib/python3.11/site-packages/geopandas/geodataframe.py:1819: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  super().__setitem__(key, value)


30227

In [30]:
missing_timesteps = ((patches["S1_pre_month_date"].isna() & patches["S2_pre_month_date"].isna()).astype(bool) |
                 (patches["S1_pre_event_date"].isna() & patches["S2_pre_event_date"].isna()).astype(bool) |
                 patches["S1_event_window_date"].isna() |
                 patches["S2_event_window_date"].isna() |
                 (patches["S1_post_event_date"].isna() & patches["S2_post_event_date"].isna()).astype(bool))

patches = patches[~missing_timesteps]
len(patches)

30108

In [31]:
# Filter out overlaps between tiles of the same tile
gdf = patches.copy()
gdf["geometry"] = gdf["geometry"].buffer(-0.005)

# Check for overlaps between tiles
overlaps = gpd.sjoin(gdf, gdf, predicate="intersects", how="inner")
# Drop itself and patches from the same tile
overlaps = overlaps[overlaps.event_id_left != overlaps.event_id_right]
# Drop samples from different events
overlaps = overlaps[(overlaps.code_left == overlaps.code_right) & (overlaps.aoi_number_left == overlaps.aoi_number_right)]
len(overlaps)

/var/folders/sr/4k1bv9r565d12nx315mt59_r0000gp/T/ipykernel_10768/2945025839.py:3: UserWarning: Geometry is in a geographic CRS. Results from 'buffer' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  gdf["geometry"] = gdf["geometry"].buffer(-0.005)


5628

In [32]:
keep = overlaps[["event_id_left", "S1_num_images_left", "S2_num_images_left", "event_id_right", "S1_num_images_right", "S2_num_images_right"]].drop_duplicates()
keep["keep"] = keep.apply(lambda row: row.S1_num_images_left + row.S2_num_images_left >= row.S1_num_images_right + row.S2_num_images_right, axis=1)
keep = keep.groupby(["event_id_left", "event_id_right"]).min().reset_index()
# Drop the samples the other way around
for i, row in keep.copy().iterrows():
    if row["event_id_left"] in keep["event_id_left"].values and row["event_id_right"] in keep[keep["event_id_left"] == row["event_id_left"]].values:
        keep = keep[~((keep["event_id_left"] == row["event_id_right"]) & (keep["event_id_right"] == row["event_id_left"]))]
len(keep)

122

In [33]:
drop = []
for i, row in overlaps.iterrows():
    left = keep[(keep['event_id_left'] == row['event_id_left']) & (keep['event_id_right'] == row['event_id_right'])]['keep']
    right = keep[(keep['event_id_right'] == row['event_id_left']) & (keep['event_id_left'] == row['event_id_right'])]['keep']
    if len(left) and not left.item() and not row["patch_id_right"] in drop:
        drop.append(i)
    elif len(right) and right.item() and not row["patch_id_right"] in drop:
        drop.append(i)
len(drop)

2814

In [34]:
patches = patches[~patches.index.isin(drop)]
len(patches)

27705

In [35]:
# Only keep samples with 2 S1 and 2 S2 images
patches = patches[(patches["S1_num_images"] >= 2) & (patches["S2_num_images"] >= 2)]
len(patches)

26782

In [36]:
# drop negative samples with more than 5% missing labels
patches = patches[~(patches["negative_sample"] & (patches["pixel_ignore"] > 3250))]
len(patches)

25253

In [37]:
patches

,patch_id,event_id,title,location,aoi_name,event_type,subcategory,event_date,annotation_date,code,...,S2_pre_month_cloud_coverage,S2_pre_event_cloud_coverage,S2_event_window_cloud_coverage,S2_post_event_cloud_coverage,pixel_ignore,pixel_background,pixel_wildfire,positive_sample,split,negative_sample
0,EMSR533_1_31SFA_x600545_y4033625,EMSR533_1_31SFA,Algeria Forest Fires,Algeria,Tizi Ouzou,Wildfire,,2021-08-11,2021-08-20,EMSR533,...,0.0,0.0,0.000000,0.0,1068,64468,0,False,train,True
1,EMSR533_1_31SFA_x603105_y4033625,EMSR533_1_31SFA,Algeria Forest Fires,Algeria,Tizi Ouzou,Wildfire,,2021-08-11,2021-08-20,EMSR533,...,0.0,0.0,0.000000,0.0,0,65536,0,False,train,True
3,EMSR533_1_31SFA_x605665_y4033625,EMSR533_1_31SFA,Algeria Forest Fires,Algeria,Tizi Ouzou,Wildfire,,2021-08-11,2021-08-20,EMSR533,...,0.0,0.0,0.000000,0.0,0,65536,0,False,train,True
4,EMSR533_1_31SFA_x605665_y4031065,EMSR533_1_31SFA,Algeria Forest Fires,Algeria,Tizi Ouzou,Wildfire,,2021-08-11,2021-08-20,EMSR533,...,0.0,0.0,0.000000,0.0,0,65536,0,False,train,True
5,EMSR533_1_31SFA_x608225_y4033625,EMSR533_1_31SFA,Algeria Forest Fires,Algeria,Tizi Ouzou,Wildfire,,2021-08-11,2021-08-20,EMSR533,...,0.0,0.0,0.000000,0.0,0,62472,3064,True,train,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
33963,EMSR647_2_18HYD_x744475_y5896105,EMSR647_2_18HYD,Forest Fires in Chile,Chile,Rafael,Wildfire,,2023-02-05,2023-02-17,EMSR647,...,0.0,0.0,0.000916,0.0,0,65536,0,False,train,True
33964,EMSR647_2_18HYD_x744475_y5893545,EMSR647_2_18HYD,Forest Fires in Chile,Chile,Rafael,Wildfire,,2023-02-05,2023-02-17,EMSR647,...,0.0,0.0,0.000000,0.0,0,65536,0,False,train,True
33965,EMSR647_2_18HYD_x747035_y5898665,EMSR647_2_18HYD,Forest Fires in Chile,Chile,Rafael,Wildfire,,2023-02-05,2023-02-17,EMSR647,...,0.0,0.0,0.000000,0.0,0,65536,0,False,train,True
33966,EMSR647_2_18HYD_x747035_y5896105,EMSR647_2_18HYD,Forest Fires in Chile,Chile,Rafael,Wildfire,,2023-02-05,2023-02-17,EMSR647,...,0.0,0.0,0.000000,0.0,0,65536,0,False,train,True


In [38]:
# Subsample negative samples
drop_codes = patches[patches["negative_sample"]].value_counts("code").index[:10]
drop = patches[(patches["negative_sample"]) & patches["code"].isin(drop_codes)].sample(frac=0.4).index.tolist()
drop_codes = patches[patches["negative_sample"]].value_counts("code").index[10:50]
drop.extend(patches[(patches["negative_sample"]) & patches["code"].isin(drop_codes)].sample(frac=0.25).index.tolist())
patches = patches.drop(drop)
len(patches)

21452

In [39]:
patches.value_counts("code")

code
EMSR765    2842
EMSR408    2022
EMSR250    1078
EMSR837    1043
EMSR213     812
           ... 
EMSR230       1
EMSR505       1
EMSR594       1
EMSR288       1
EMSR430       1
Name: count, Length: 242, dtype: int64

# Subsample fire event

In [40]:
# Subsample fire event EMSR765
fire_sel = patches.copy()
len(fire_sel)

21452

In [41]:
import numpy as np
import pandas as pd

def binary_label_entropy_from_counts(bg, fire, eps=1e-12):
    """
    Shannon entropy H(p) for a binary label distribution (background vs wildfire).
    bg: number of background pixels
    fire: number of wildfire pixels
    Returns entropy in [0,1] (base-2 normalization).
    """
    bg = pd.Series(bg).astype(float)
    fire = pd.Series(fire).astype(float)
    total = bg + fire
    # If no labeled pixels, entropy = 0 (uninformative)
    with np.errstate(divide='ignore', invalid='ignore'):
        p = np.where(total > 0, fire / total, 0.0)
    p = np.clip(p, 0.0 + eps, 1.0 - eps)  # avoid log(0)
    H = -(p * np.log2(p) + (1 - p) * np.log2(1 - p))
    # Max entropy for binary is 1 (when p=0.5)
    return pd.Series(H, index=bg.index)

def combine_nan_ratio(df, s1_col='S1_nan_ratio', s2_col='S2_nan_ratio', mode='weighted', w_s1=0.5):
    """
    Combine NaN ratios from S1 and S2 into a single value in [0,1].
    mode='max'|'mean'|'weighted' (weighted mean).
    """
    s1 = pd.to_numeric(df.get(s1_col, pd.Series(np.nan, index=df.index)), errors='coerce')
    s2 = pd.to_numeric(df.get(s2_col, pd.Series(np.nan, index=df.index)), errors='coerce')
    if mode == 'max':
        combined = pd.concat([s1, s2], axis=1).max(axis=1, skipna=True)
    elif mode == 'mean':
        combined = pd.concat([s1, s2], axis=1).mean(axis=1, skipna=True)
    else:
        # weighted mean: w_s1 * S1 + (1-w_s1) * S2, falling back gracefully
        combined = w_s1 * s1.fillna(s2) + (1 - w_s1) * s2.fillna(s1)
    return combined.clip(0.0, 1.0)

def compute_sampling_probabilities(
    df,
    cloud_col='S2_cloud_coverage',
    nan_mode=('weighted', 0.4),   # (mode, w_s1)
    entropy_col=None,             # if None, compute from bg/fire counts
    entropy_from_counts=('pixel_background', 'pixel_wildfire'),
    alphas=(1.0, 1.0, 1.0),       # (alpha_cloud, alpha_nan, alpha_entropy)
    temperature=1.0,
    min_prob=1e-9
):
    """
    Returns a pandas Series of probabilities aligned with df.index.

    - alphas: exponents for (1-cloud), (1-nan), entropy after robust scaling.
    - temperature: <1 sharpens, >1 flattens distribution.
    - min_prob: floor added before normalization to avoid zeros.
    """
    # Cloud
    cloud = pd.to_numeric(df.get(cloud_col, pd.Series(np.nan, index=df.index)), errors='coerce').fillna(1.0)
    cloud = cloud.clip(0.0, 1.0)

    # NaN
    if isinstance(nan_mode, tuple):
        mode, w_s1 = nan_mode
    else:
        mode, w_s1 = 'weighted', 0.5
    nan_combined = combine_nan_ratio(df, mode=mode, w_s1=w_s1).fillna(1.0).clip(0.0, 1.0)

    # Entropy (prefer higher)
    if entropy_col is not None and entropy_col in df.columns:
        entropy = pd.to_numeric(df[entropy_col], errors='coerce').fillna(0.0)
        # If entropy not normalized, scale robustly to [0,1]
    else:
        # Compute from label counts
        bg_col, fire_col = entropy_from_counts
        entropy = binary_label_entropy_from_counts(df.get(bg_col, 0) + 100, df.get(fire_col, 0) + 100)
        entropy = entropy.fillna(0.0)

    aoi_count = 1 / df.value_counts('aoi_number') * 50
    aoi = df['aoi_number'].apply(lambda x: aoi_count[x])

    split_count = 1 / df.value_counts('split') * 1000
    split_weight = df['split'].apply(lambda x: split_count[x])

    # Exponents
    a_cloud, a_nan, a_ent = alphas
    with np.errstate(invalid='ignore'):
        score = (cloud.clip(0,1) ** a_cloud) + (nan_combined.clip(0,1) ** a_nan) + (entropy.clip(0,1) ** a_ent) + aoi + split_weight

    # Temperature: T<1 → sharper, T>1 → flatter
    T = max(float(temperature), 1e-6)
    score = np.power(score + 1e-12, 1.0 / T)

    # Floor and normalize
    score = score + float(min_prob)
    total = score.sum()
    if not np.isfinite(total) or total <= 0:
        # Fallback to uniform probs
        probs = pd.Series(np.ones(len(df)) / len(df), index=df.index)
    else:
        probs = score / total

    return probs

def probabilistic_subsample(
    gdf,
    n=None,
    frac=None,
    random_state=42,
    groupby=None,             # e.g., 'event_id' to enforce per-group quotas
    per_group=None,           # dict: group -> n, or int for uniform per-group
    **prob_kwargs
):
    """
    Weighted random subsample without replacement.
    Either set `n` or `frac`. Optionally enforce per-group quotas.

    prob_kwargs are passed to compute_sampling_probabilities().
    """
    assert (n is None) ^ (frac is None), "Specify exactly one of n or frac."

    if groupby is None and per_group is None:
        probs = compute_sampling_probabilities(gdf, **prob_kwargs)
        return gdf.sample(n=n, frac=frac, weights=probs, random_state=random_state, replace=False)

    # Group-aware sampling
    if groupby is not None:
        groups = gdf.groupby(groupby)
        sampled_parts = []
        probs_all = compute_sampling_probabilities(gdf, **prob_kwargs)

        if per_group is None:
            # Split target equally by group sizes (by frac) or proportionally (by n)
            if frac is not None:
                for _, sub in groups:
                    k = max(1, int(np.floor(len(sub) * frac)))
                    sampled_parts.append(sub.sample(n=k, weights=probs_all.loc[sub.index], random_state=random_state, replace=False))
            else:
                sizes = groups.size()
                w = sizes / sizes.sum()
                for gid, sub in groups:
                    k = int(np.floor(w.loc[gid] * n))
                    if k > 0:
                        sampled_parts.append(sub.sample(n=k, weights=probs_all.loc[sub.index], random_state=random_state, replace=False))
        else:
            # Fixed per-group k
            for gid, sub in groups:
                k = per_group if isinstance(per_group, int) else per_group.get(gid, 0)
                if k > 0 and len(sub) > 0:
                    k = min(k, len(sub))
                    sampled_parts.append(sub.sample(n=k, weights=probs_all.loc[sub.index], random_state=random_state, replace=False))

        out = pd.concat(sampled_parts, axis=0) if sampled_parts else gdf.iloc[[]]
        return out.sample(frac=1.0, random_state=random_state)  # shuffle

    # If per_group provided without groupby
    raise ValueError("If per_group is set, please also set groupby.")

sampled_765 = probabilistic_subsample(
    fire_sel[fire_sel.code == "EMSR765"],
    n=2000,                 # or use frac=0.1
    random_state=42,
    # Optional per-event quotas:
    # groupby='event_id', per_group=50,
    cloud_col='S2_cloud_coverage',
    nan_mode=('weighted', 0.4),    # 40% S1, 60% S2
    entropy_col=None,              # compute from counts below
    entropy_from_counts=('pixel_background', 'pixel_wildfire'),
    alphas=(1.0, 1.2, 1.0),        # slightly stronger penalty for NaNs
    temperature=1,               # sharper preferences
    min_prob=1e-8
)

In [42]:
sampled_765.value_counts('aoi_number')
# fire_sel[fire_sel.code == "EMSR765"].value_counts('aoi_number')

aoi_number
13    609
3     275
1     252
5     251
9     164
6     160
7     147
10    102
14     40
Name: count, dtype: int64

In [43]:
# Random sampling
# sampled_765 = fire_sel[fire_sel.code == "EMSR765"].sample(n=5000).index

In [44]:
fire_sel = fire_sel[(fire_sel.code != "EMSR765") | fire_sel.patch_id.isin(sampled_765.patch_id.values)]
len(fire_sel)

20610

In [45]:
# Patches per split
fire_sel.value_counts('split') / len(fire_sel)

split
train    0.705822
test     0.163950
val      0.130228
Name: count, dtype: float64

In [46]:
# Events per split
fire_sel.groupby(["split"]).value_counts(["code"]).groupby(level=0).count()

split
test      72
train    169
val       59
Name: count, dtype: int64

In [47]:
# Reset
fire_sel.drop(columns='negative_sample', inplace=True)
fire_sel.reset_index(drop=True, inplace=True)

In [48]:
fire_tile_sel = fire.copy()
# Update num patches
fire_tile_sel.set_index('event_id', inplace=True, drop=True)
fire_tile_sel["num_patches"] = fire_sel.value_counts('event_id')
fire_tile_sel["num_pos_patches"] = fire_sel[fire_sel["positive_sample"]].value_counts('event_id')
# Drop tiles without patches
fire_tile_sel = fire_tile_sel.dropna(subset=['num_patches'])
fire_tile_sel[["num_patches", "num_pos_patches"]] = fire_tile_sel[["num_patches", "num_pos_patches"]].fillna(0).astype(int)
# Update geometry based on patches
for g, group in fire_sel.groupby('event_id'):
    fire_tile_sel.at[g, "geometry"] = group.union_all()
fire_tile_sel = fire_tile_sel.drop(columns='aoi').set_geometry('geometry')
fire_tile_sel.reset_index(inplace=True)
fire_tile_sel = fire_tile_sel.set_crs(4326)

In [49]:
print(len(fire_tile_sel.code.unique()), "events")

242 events


In [50]:
fire_aoi_sel = fire_tile_sel.drop_duplicates(['code', 'aoi_number'])
fire_aoi_sel.set_index(['code', 'aoi_number'], inplace=True, drop=True)
fire_aoi_sel["num_patches"] = fire_sel.value_counts(['code', 'aoi_number'])
fire_aoi_sel["num_pos_patches"] = fire_sel[fire_sel["positive_sample"]].value_counts(['code', 'aoi_number'])
fire_aoi_sel[["num_patches", "num_pos_patches"]] = fire_aoi_sel[["num_patches", "num_pos_patches"]].fillna(0).astype(int)
# Update geometry based on patches
for g, group in fire_sel.groupby(['code', 'aoi_number']):
    fire_aoi_sel.at[g, "geometry"] = group.union_all()

fire_aoi_sel.reset_index(inplace=True)
fire_aoi_sel = fire_aoi_sel.set_crs(4326)

/Users/BLU/repos/impactmesh/venv/lib/python3.11/site-packages/geopandas/geodataframe.py:1819: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  super().__setitem__(key, value)
/Users/BLU/repos/impactmesh/venv/lib/python3.11/site-packages/geopandas/geodataframe.py:1819: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  super().__setitem__(key, value)
/Users/BLU/repos/impactmesh/venv/lib/python3.11/site-packages/geopandas/geodataframe.py:1819: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice 

In [51]:
fire_sel.to_parquet("../data/CCC-ImpactMesh-Fire/wildfire_patch_metadata.parquet")
fire_tile_sel.to_parquet("../data/CCC-ImpactMesh-Fire/wildfire_tile_metadata.parquet")
fire_tile_sel.to_parquet("../data/CCC-ImpactMesh-Fire/wildfire_aoi_metadata.parquet")

# fire = gpd.read_parquet("../data/AD-ImpactMesh-Fire/tile_metadata.parquet").reset_index(drop=True)
# fire_patches = gpd.read_parquet("../data/AD-ImpactMesh-Fire/patch_metadata.parquet").reset_index(drop=True)

In [9]:
fire = gpd.read_parquet("../data/CCC-ImpactMesh-Fire/tile_metadata.parquet").reset_index(drop=True)

In [13]:
fire[fire.event_date.isna()]['event_date'] = pd.to_datetime('16/01/2026', dayfirst=True)

/Users/BLU/repos/impactmesh/venv/lib/python3.11/site-packages/geopandas/geodataframe.py:1819: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  super().__setitem__(key, value)


In [21]:
fire_sel['event_date'] = fire_sel['event_date'].fillna(pd.to_datetime('16/01/2026', dayfirst=True))

In [3]:
fire_sel = gpd.read_parquet("../data/CCC-ImpactMesh-Fire/fire_patch_metadata_updated.parquet")
fire_tile_sel = gpd.read_parquet("../data/CCC-ImpactMesh-Fire/fire_tile_metadata_updated.parquet")

In [22]:
fire_sel.to_parquet("../data/CCC-ImpactMesh-Fire/fire_patch_metadata_updated.parquet")

In [23]:
events = pd.read_parquet('../data/new/new_events.parquet')

In [26]:
events[events.code == 'EMSR859']

,event_id,code,title,AOINumber,AOIName,activation_date,location,event_type,subCategory,tile_name,...,AnnZip,geometry,continent,activator,closed,sensitive,product_type,monitoring,monitoring_number,product_version
305,EMSR859_1_18HYD,EMSR859,"Wildfires in the Biobio and Nuble Regions, Chile",1,San Rosendo,2026-01-21 03:02:00,Chile,Wildfire,Forest fire,18HYD,...,[/Users/BLU/repos/impactmesh/data/new/cems_ann...,b'\x01\x03\x00\x00\x00\x01\x00\x00\x00\x0b\x00...,South America,EC Services|DG ECHO,True,False,GRA,False,0,3
306,EMSR859_2_18HYD,EMSR859,"Wildfires in the Biobio and Nuble Regions, Chile",2,Chiguaihue,2026-01-21 03:02:00,Chile,Wildfire,Forest fire,18HYD,...,[/Users/BLU/repos/impactmesh/data/new/cems_ann...,b'\x01\x03\x00\x00\x00\x01\x00\x00\x00\x0f\x00...,South America,EC Services|DG ECHO,True,False,DEL,True,3,2
307,EMSR859_2_18HYC,EMSR859,"Wildfires in the Biobio and Nuble Regions, Chile",2,Chiguaihue,2026-01-21 03:02:00,Chile,Wildfire,Forest fire,18HYC,...,[/Users/BLU/repos/impactmesh/data/new/cems_ann...,b'\x01\x03\x00\x00\x00\x01\x00\x00\x00\x0f\x00...,South America,EC Services|DG ECHO,True,False,DEL,True,3,2
308,EMSR859_3_18HYE,EMSR859,"Wildfires in the Biobio and Nuble Regions, Chile",3,Santa Barbara,2026-01-21 03:02:00,Chile,Wildfire,Forest fire,18HYE,...,[/Users/BLU/repos/impactmesh/data/new/cems_ann...,b'\x01\x03\x00\x00\x00\x01\x00\x00\x00\x14\x00...,South America,EC Services|DG ECHO,True,False,DEL,False,0,2
309,EMSR859_3_18HXE,EMSR859,"Wildfires in the Biobio and Nuble Regions, Chile",3,Santa Barbara,2026-01-21 03:02:00,Chile,Wildfire,Forest fire,18HXE,...,[/Users/BLU/repos/impactmesh/data/new/cems_ann...,b'\x01\x03\x00\x00\x00\x01\x00\x00\x00\x14\x00...,South America,EC Services|DG ECHO,True,False,DEL,False,0,2
310,EMSR859_4_18HXE,EMSR859,"Wildfires in the Biobio and Nuble Regions, Chile",4,Conception,2026-01-21 03:02:00,Chile,Wildfire,Forest fire,18HXE,...,[/Users/BLU/repos/impactmesh/data/new/cems_ann...,b'\x01\x03\x00\x00\x00\x01\x00\x00\x00\x0e\x00...,South America,EC Services|DG ECHO,True,False,DEL,True,2,1


In [41]:
# # Drop patches with errors
# import re
# error_patches = ["EMSR408_7_56JLL_x406835*", "EMSR765_13_20KRG_x883425*", "EMSR837_4_29TPH_x701805*", "EMSR760_4_29TNG_*_y4603685", "EMSR647_1_18HXD_*_y5876605", "EMSR765_6_20LLJ_x396015_*", "EMSR765_9_20LKH_x307005_y8242915"]
#
# def wildcard_to_regex(pattern):
#     pattern = re.escape(pattern)
#     pattern = pattern.replace(r'\*', '.*')
#     return f'^{pattern}$'
#
# regex_patterns = [wildcard_to_regex(p) for p in error_patches]
# combined_pattern = '|'.join(regex_patterns)
# fire_sel = fire_sel[~fire_sel['patch_id'].str.match(combined_pattern, na=False)]
# fire_sel.to_parquet("../data/CCC-ImpactMesh-Fire/fire_patch_metadata_updated.parquet")

In [43]:
fire_sel = fire_sel.sample(frac=1)

In [44]:
out_dir = Path("../data/CCC-ImpactMesh-Fire/split/")
out_dir.mkdir(parents=True, exist_ok=True)
for s, group in fire_sel.groupby("split"):
    print(s, len(group))
    with open(out_dir / f"impactmesh_fire_{s}.txt", "w") as f:
        f.write("\n".join(group.patch_id.tolist()))
        f.write("\n")

test 2896
train 12385
val 2227


In [45]:
# Get geographic hold out set
train_codes = fire_sel[(fire_sel['split'] == 'train')].code.unique()
holdout = fire_sel[(fire_sel['split'] == 'test') & ~fire_sel['code'].isin(train_codes)]
print(len(holdout))

with open(out_dir / f"impactmesh_fire_test_holdout.txt", "w") as f:
    f.write("\n".join(holdout.patch_id.tolist()))
    f.write("\n")

1384


In [46]:
out_dir = Path("../data/CCC-ImpactMesh-Fire/split/")
out_dir.mkdir(parents=True, exist_ok=True)
for s, group in fire_tile_sel.groupby("split"):
    print(s, len(group))
    with open(out_dir / f"impactmesh_fire_tile_{s}.txt", "w") as f:
        f.write("\n".join(group.event_id.tolist()))
        f.write("\n")

test 105
train 361
val 84


In [47]:
split_colors = {
    "train": "#198038",  # dark green
    "val":   "#F09235",  # dark orange
    "test":  "#922752",  # dark red
    "test (holdout events)":  "#922752",  # dark red
}

split_edge = {
    "train": "#198038",  # dark green
    "val":   "#F09235",  # dark orange
    "test":  "#922752",  # dark red
    "test (holdout events)":  "#111111",  # dark red
}

def _escape_html(s: str) -> str:
    return (
        str(s)
        .replace("&", "&amp;")
        .replace("<", "&lt;")
        .replace(">", "&gt;")
        .replace('"', "&quot;")
        .replace("'", "&#39;")
    )

def _build_legend_html(edge_colors: dict, fill_colors: dict) -> str:
    """Builds a small HTML legend showing stroke colors by event type and fill colors by split."""
    edge_rows = "".join(
        f"<div style='display:flex; align-items:center; gap:6px; margin:2px 0;'>"
        f"<span style='display:inline-block; width:16px; height:14px; "
        f"border:3px solid {edge}; background:{fill_colors[label]};'></span>"
        f"<span>{_escape_html(label)}</span></div>"
        for label, edge in edge_colors.items()
    )

    return f"""
    <div style="
        position: fixed;
        bottom: 18px;
        left: 18px;
        z-index: 9999;
        background: white;
        border: 2px solid #444;
        border-radius: 6px;
        padding: 10px 12px;
        font-family: Arial, sans-serif;
        font-size: 12px;
        box-shadow: 0 1px 6px rgba(0,0,0,0.25);
    ">
      <div style="font-weight: 700; margin-bottom: 6px;">Legend</div>
      <div style="margin-bottom: 8px;">
        {edge_rows}
      </div>
    </div>
    """

def map_from_gdf(gdf):
    m = folium.Map()
    folium.GeoJson(
        gdf,
        style_function=lambda x: {
            'color': split_edge[x["properties"]["split"]],
            'weight': 2,
            # 'fillColor': f'#{int(x["properties"]["cell_id"]) * 123456 % 0xFFFFFF:06x}',
            'fillColor': split_colors[x["properties"]["split"]],
            'fillOpacity': 0.6
        },
        tooltip=GeoJsonTooltip(fields=("title", "code", "aoi_name", "location", "num_patches", "num_pos_patches", "split"))
    ).add_to(m)
    minx, miny, maxx, maxy = gdf.total_bounds
    m.fit_bounds([[miny/2, minx/2], [maxy/2, maxx/2]])

    legend_html = _build_legend_html(split_edge, split_colors)
    m.get_root().html.add_child(folium.Element(legend_html))

    return m


# Events per split
holdout_codes = holdout.code.unique()
aoi_copy = fire_aoi_sel[["title", "code", "aoi_name", "location", "num_patches", "num_pos_patches", "split", "geometry"]]
aoi_copy['split'] = aoi_copy.apply(lambda row: row.split if row.code not in holdout_codes else "test (holdout events)", axis=1)

m = map_from_gdf(aoi_copy)
m.save("../data/CCC-ImpactMesh-Fire/wildfire_split_map.html")
m

NameError: name 'fire_aoi_sel' is not defined